In [10]:
# import packages
import osgeo
import xarray as xr
import geopandas as gpd
import rioxarray as rxr
import odc.geo.xr

from pathlib import Path

import warnings
warnings.filterwarnings("ignore")

In [11]:
# Load configuration
%run 00_config.ipynb
print_config()

CURRENT CONFIGURATION

Date Range:
  Start: 2024-01-01
  End: 2024-12-31

Processing Settings:
  Sample Period: 1ME
  Statistical Method: mean
  Region: All
  Pollutants: SO2, NO2, CO, O3, PM2P5, PM10

Output Path:
  C:\projects\aqi_scad\03_Output_Data\monthly_data


In [12]:
# Optionally Use dask package to increase performance
# from dask.distributed import Client
# client = Client()  # set up local cluster on your computer
# client

In [13]:
# General_Functions
%run functions.ipynb

In [14]:
# Path to Region Boundary File
boundary_vector = PATHS['boundary_vector']
# load boundaries into geodataframe
gdf = gpd.read_file(boundary_vector).to_crs("EPSG:3857")
# Field housing name of region
region_field = 'REGIONNAME'

In [15]:
# Data Path
input_path =  PATHS['input']

# Selected Pollutant
pollutants = ['SO2', 'NO2', 'CO', 'PM2P5', 'PM10', 'O3']

# Start and End Dates
start_date = '2024-01-01'
end_date = '2024-12-31'

# Sampling Frequency/Period
sample_period = '1ME'

# Extract Region (Options are 'All','Al Ain', Al Dhafra, 'Abu Dhabi')
region = 'All'

if region == 'All':
    aoi = gdf
else:
    aoi = gdf[gdf[region_field] == region]

# Statistics Method ('mean', 'median', 'max')
stat_method = 'mean'

# Output Locations
output_folder =  PATHS['aqi_data']
output_folder.mkdir(parents=True, exist_ok=True)

In [16]:
aqi_da_list = []
for pollutant in pollutants:
    data_array = load_data(data_path = input_path, pollutant = pollutant, region= aoi)
    data_array_temporal = resample_data(data_array = data_array, start_date=start_date,
                                        end_date=end_date, sample_period=sample_period, stat_method=stat_method)
    pollutant_aqi = calculate_pollutant_aqi(data_array = data_array_temporal, pollutant = pollutant)
    aqi_da_list.append(pollutant_aqi)

aqi_array = xr.concat(aqi_da_list, dim='time')
aqi_array = aqi_array.compute()

Loading: C:\projects\aqi_scad\02_Input\CAMS_SO2_ugm3.zarr
  - RAW data range: 0.06 to 5.53
  - Time range: 1970-01-01T00:00:00.000000000 to 1970-01-01T00:00:00.000000000
  - Valid positive values: 250333632
Loading: C:\projects\aqi_scad\02_Input\MOD_SO2_ugm3.zarr
  - RAW data range: nan to nan
  - Time range: 2022-01-01T00:00:00.000000000 to 2022-01-01T00:00:00.000000000
  - Valid positive values: 0
Loading: C:\projects\aqi_scad\02_Input\S5P_SO2_ugm3.zarr
  - RAW data range: -64.06 to 599.49
  - Time range: 2022-01-01T01:08:13.000000000 to 2024-12-30T22:55:03.000000000
  - Valid positive values: 95570908


AlignmentError: cannot align objects with join='override' with matching indexes along dimension 'x' that don't have the same size

In [7]:
# Calculate Monhtly AQI
aqi_array_1d = aqi_array.sortby('time').resample(time='1ME').max()
aqi_array_ds = aqi_array_1d.to_dataset(name='AQI')


In [8]:
aqi_array_ds= aqi_array_ds.rio.write_crs(3857)
aqi_array_ds = aqi_array_ds.rio.clip(aoi.geometry.values, aoi.crs, all_touched=True, drop=False)

In [9]:
# Export's Data as Geotiff
identifiers = [i for i in range(len(aqi_array_ds.time+1))]
export_data_tif(aqi_array_ds, output_folder, identifiers, 'AQI', sample_period, stat_method)